# SAC Irrigation Training — v2.9.0 (Colab Pro)

**Architecture:** CTDE + VDN Factorized Critic — v2.7 (1097-dim obs, 8 features/agent)  
**Single change vs v2.7:** `AdaptiveLRCallback` (Proposal B) added to callback stack  
**Hyperparameters:** `ent_coef=0.05` (fixed), `max_grad_norm=1.0`, LR decay 3e-4→5e-5

## What v2.9 tests
v2.7 and v2.8 both exhibit a critic-loss explosion at step ~165–195k (the SAC  
deadly-triad: overestimated Q → actor exploitation → divergence).  
v2.9 adds **Proposal B**: a callback that detects the rising critic_loss and reduces LR  
by 0.7× before the cascade ignites.  Recovery logic restores the scheduled LR once  
critic_loss settles.  Everything else is identical to v2.7.

| # | Change vs v2.7 | Effect |
|---|---|---|
| 1 | `AdaptiveLRCallback` | Fires when rolling-1000-step critic_loss > 50 → LR ×0.3; recovers when < 5 |

## What is NOT changed vs v2.7
- Observation layout: 1097-dim, 8 features/agent (no `x1_overshoot_norm`)
- No episode-length curriculum (full 93-day episodes from step 0)
- All reward terms, SAC hyperparameters, architecture identical to v2.7

## Before running
1. `Runtime → Change runtime type → A100 GPU`
2. Add WandB API key: sidebar → key icon → `WANDB_API_KEY` → Notebook access ON
3. Mount Google Drive in Cell 1

## What to watch on WandB
| Metric | v2.7 behaviour | v2.9 success looks like | Abort if |
|--------|----------------|------------------------|----------|
| `train/critic_loss` | Spikes to 1e6+ at step ~165k | Stays < 200 throughout; brief spikes are damped | Exceeds 1e4 after step 200k |
| `adaptive_lr/event` | N/A | One or two events around step 160–200k | >5 events (pathological oscillation) |
| `adaptive_lr/lr_after_reduce` | N/A | ~9e-5 (= 3e-4 × 0.3) | — |
| `eval/mean_reward` | Peaks at step 200k | Peaks same or later | Monotonically decreasing after step 100k |

## Falsifiable predictions
- **If Proposal B works:** critic_loss bounded < 200; best_model step 200–250k; yield same or better vs v2.7
- **If Proposal B doesn't work:** same explosion pattern as v2.7 shifted slightly later; best_model step ~200k; same yields

## Seed plan (one seed this session — decide on more after results)
Use `SEED = 0` (paired with v2.7 seed 0 corrected baseline).
Results saved to `MyDrive/thesis_results/sac_v29_seed{N}_{timestamp}/`.

## Estimated runtimes (identical obs dim to v2.7)
| Hardware | Steps/sec | 250k steps |
|----------|-----------|------------|
| A100 | ~120–160 | ~26–35 min |
| L4 | ~75–95 | ~44–55 min |
| T4 | ~55–75 | ~55–75 min |

In [ ]:
# ── CELL 1: Mount Drive, clone repo, install deps ───────────────────────────
import subprocess, sys, os

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = '/content/drive/MyDrive/thesis_results'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'✓  Drive mounted. Results → {DRIVE_ROOT}')

if os.path.exists('/content/thesis'):
    subprocess.run(['rm', '-rf', '/content/thesis'], check=True)
subprocess.run(
    ['git', 'clone', 'https://github.com/taratorbati/thesis.git', '/content/thesis'],
    check=True
)
os.chdir('/content/thesis')
sys.path.insert(0, '/content/thesis')
print('✓  Repo cloned')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'stable-baselines3==2.6.0',
    'gymnasium==1.1.1',
    'wandb>=0.16',
    'pytest',
], check=True)

import numpy as np, gymnasium, stable_baselines3 as sb3, torch
print(f'numpy:             {np.__version__}')
print(f'gymnasium:         {gymnasium.__version__}')
print(f'stable-baselines3: {sb3.__version__}')
print(f'torch:             {torch.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:               {torch.cuda.get_device_name(0)}')
torch.set_num_threads(4)

In [ ]:
# ── CELL 2: WandB secret + GPU check ────────────────────────────────────────
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('✓  WANDB_API_KEY loaded from Colab Secrets.')
except Exception as e:
    print(f'⚠  Could not load WANDB_API_KEY ({type(e).__name__}).')
    print('   Training continues without WandB — add key to Colab Secrets to enable.')

import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'nvidia-smi failed — no GPU allocated')

In [ ]:
# ── CELL 3: Pre-training validation (MUST PASS before Cell 4) ───────────────
#
# Runs smoke tests, VDN unit tests, adaptive-LR callback self-test, and bench.
# DO NOT proceed to Cell 4 if any test fails.

import subprocess, sys, time

print('Running smoke tests (includes obs-parity test v2.8.1)...')
r = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_rl_smoke.py', '-v', '--tb=short'],
    capture_output=False
)
assert r.returncode == 0, 'SMOKE TESTS FAILED'
print()

print('Running VDN unit tests...')
r2 = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_factorized_critic.py', '-v', '--tb=short'],
    capture_output=False
)
assert r2.returncode == 0, 'VDN UNIT TESTS FAILED'
print()

# ── Adaptive LR callback self-test ─────────────────────────────────────────
print('Testing AdaptiveLRCallback (Proposal B)...')
from src.rl.train import AdaptiveLRCallback, _make_lr_schedule, LR_START, LR_END
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from src.rl.gym_env import IrrigationEnv
from src.rl.networks import V27CTDESACPolicy, make_sac_policy_kwargs

# Instantiate a tiny model to verify the callback can reach optimizer params
_test_env = DummyVecEnv([lambda: IrrigationEnv(randomize=True, curriculum_warmup_steps=0)])
_test_model = SAC(
    policy=V27CTDESACPolicy, env=_test_env,
    policy_kwargs=make_sac_policy_kwargs(N=130),
    buffer_size=2_000, batch_size=64, learning_starts=200,
    ent_coef=0.05, verbose=0, seed=0,
)
lr_sched = _make_lr_schedule(LR_START, LR_END)
_cb = AdaptiveLRCallback(lr_schedule=lr_sched, verbose=0)
_cb.init_callback(_test_model)
# Set a fake current LR and simulate a spike detection
for pg in _test_model.policy.actor.optimizer.param_groups:
    pg['lr'] = 3e-4
import numpy as np
_cb._critic_loss_buf.extend([100.0] * _cb.window)  # force spike condition
_cb._last_reduction_step = -10_000
_cb.num_timesteps = 5_000
# Manually trigger the spike branch
from collections import deque
import numpy as _np
rolling = float(_np.mean(_cb._critic_loss_buf))
assert rolling > _cb.spike_threshold, 'Test buffer did not reach spike threshold'
current_lr = _cb._get_lr()
new_lr = max(current_lr * _cb.reduction_factor, _cb.lr_floor)
_cb._set_lr(new_lr)
assert abs(_cb._get_lr() - 9e-5) < 1e-10, (
    f'LR after spike should be 3e-4 * 0.3 = 9e-5, got {_cb._get_lr():.3e}')
del _test_env, _test_model, _cb
print('AdaptiveLRCallback self-test: PASSED  (spike: 3e-4 → 9e-5 ✓)')
print()

# ── Step rate bench ────────────────────────────────────────────────────────
print('Benchmarking step rate...')
bench_env = DummyVecEnv([lambda: IrrigationEnv(randomize=True, curriculum_warmup_steps=0)])
bench_model = SAC(
    policy=V27CTDESACPolicy, env=bench_env,
    policy_kwargs=make_sac_policy_kwargs(N=130),
    buffer_size=5_000, batch_size=256, learning_starts=500,
    ent_coef=0.05, verbose=0, seed=0,
)
bench_model.learn(total_timesteps=300)
t0 = time.time()
bench_model.learn(total_timesteps=500, reset_num_timesteps=False)
rate = 500 / (time.time() - t0)
del bench_model, bench_env
print(f'Step rate:    {rate:.1f} steps/sec')
print(f'Est. 250k:    {250_000 / rate / 60:.0f} min  ({250_000 / rate / 3600:.1f} h)')
print()

# ── obs dimension sanity check (v2.9 uses 1097, not 1227) ─────────────────
from src.rl.gym_env import IrrigationEnv
env_check = IrrigationEnv(randomize=False)
obs_check, _ = env_check.reset()
assert obs_check.shape[0] == 1097, (
    f'Expected obs_dim=1097 for v2.9 (v2.7 architecture), got {obs_check.shape[0]}.\n'
    f'Check that you have not accidentally loaded the v2.8 gym_env.')
print(f'obs_dim:      {obs_check.shape[0]} ✓  (8 features × 130 agents + 9 scalars + 48 forecast)')
print()
print('✓  ALL VALIDATION PASSED — safe to proceed to Cell 4')

In [ ]:
# ── CELL 4: Training (250k steps, ~30-55 min on A100) ────────────────────────
#
# v2.9: v2.7 architecture + Proposal B (AdaptiveLRCallback).
# No curriculum, no x1_overshoot feature.
#
# Change SEED per session.  Start with SEED=0 (paired with corrected v2.7 seed 0).
# Results go to results/rl/sac_v29_seed{N}/ — do NOT overwrite v2.7 or v2.8 results.

SEED = 0   # ← CHANGE per session: 0, then 1 if v2.9 seed 0 shows improvement

from src.rl.train import train_sac

model = train_sac(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
)
print(f'\n✓  Training complete — seed {SEED}, v2.9')

In [ ]:
# ── CELL 5: Copy results to Google Drive ─────────────────────────────────────
import shutil, os, datetime

src = f'/content/thesis/results/rl/sac_v29_seed{SEED}'
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'{DRIVE_ROOT}/sac_v29_seed{SEED}_{timestamp}'

shutil.copytree(src, dst)
print(f'Saved to: {dst}')
print()
print('Contents:')
for f in sorted(os.listdir(dst)):
    size = os.path.getsize(os.path.join(dst, f)) if os.path.isfile(os.path.join(dst, f)) else 0
    print(f'  {f}  ({size/1024:.1f} KB)' if size > 0 else f'  {f}/')

In [ ]:
# ── CELL 6: Post-training diagnostics (v2.9)  ───────────────────────────────
#
# Runs a wet/100% episode with the best_model and computes:
#
#   spatial_std          > 0.01   (v2.7 ~0.002-0.010)
#   corr(u, rain_today)  < 0      (v2.7 s0: -0.49 wet/100%)
#   corr(u, x1)_pooled   < 0      (v2.7 s0: +0.05 wet/100% — want improvement)
#   waterlog days/agent  < 70     (v2.7 s0: 76.4 wet/100%)
#   ep_len               == 93    (no curriculum so should always be 93)
#
# PRIMARY QUESTION: Does corr(u,x1) improve vs v2.7 when the critic is stable?
# SECONDARY QUESTION: Does critic_loss stay bounded throughout training?
#   (check the WandB adaptive_lr/event chart — how many reductions occurred?)
#
# Note: x1_norm decoded from obs uses (x1-WP)/(FC-WP) formula (v2.8.1 fix).

import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv
from src.rl.gym_env import IrrigationEnv, _FC_MM, _WP_MM
import glob

# ── load model ────────────────────────────────────────────────────────────────
best_model_path = f'/content/thesis/results/rl/sac_v29_seed{SEED}/best_model/best_model'
print(f'Loading: {best_model_path}')

from src.rl.networks import V27CTDESACPolicy
eval_env = DummyVecEnv([lambda: IrrigationEnv(randomize=False, curriculum_warmup_steps=0)])
loaded   = SAC.load(best_model_path, env=eval_env,
                    custom_objects={'policy_class': V27CTDESACPolicy})

# Force wet/100% episode
try:
    from climate_data import load_cleaned_data, extract_scenario
    from soil_data import get_crop
    from src.precompute import get_precomputed
    inner = eval_env.envs[0]
    df   = load_cleaned_data()
    crop = get_crop('rice')
    obs  = eval_env.reset()
    inner._year      = 2024
    inner._budget_mm = 484.0
    inner._climate   = extract_scenario(df, 2024, crop)
    inner._precomp   = get_precomputed('wet', 'rice')
except Exception as e:
    print(f'Warning: could not force wet year ({e}); running with default.')
    obs = eval_env.reset()

# ── collect episode ───────────────────────────────────────────────────────────
all_actions_per_agent = []
all_x1_per_agent      = []
all_rain_daily        = []
ep_len = 0; done = False

while not done:
    action, _ = loaded.predict(obs, deterministic=True)
    all_actions_per_agent.append(action[0].copy())
    raw_obs = obs[0]  # (1097,) v2.9/v2.7
    # Per-agent block: 8 features x 130 agents, agent-major
    agent_grid  = raw_obs[:1040].reshape(130, 8)
    x1_norm_arr = agent_grid[:, 0]
    x1_agents   = x1_norm_arr * (_FC_MM - _WP_MM) + _WP_MM
    all_x1_per_agent.append(x1_agents)
    all_rain_daily.append(float(raw_obs[1044]))  # rain at scalar index 4
    obs, _, done_arr, _ = eval_env.step(action)
    ep_len += 1; done = done_arr[0]

# ── metrics ───────────────────────────────────────────────────────────────────
actions_arr = np.array(all_actions_per_agent)   # (T, 130)
x1_arr      = np.array(all_x1_per_agent)        # (T, 130)
rain_arr    = np.array(all_rain_daily)           # (T,)
T, N        = actions_arr.shape
u_mm_arr    = actions_arr * 12.0

u_daily      = u_mm_arr.mean(axis=1)
spatial_std  = actions_arr.mean(axis=0).std()
corr_rain    = float(np.corrcoef(u_daily, rain_arr)[0, 1])

u_pool  = u_mm_arr.flatten()
x1_pool = x1_arr.flatten()
corr_x1_pooled = float(np.corrcoef(u_pool, x1_pool)[0, 1])

waterlog_days = float((x1_arr > _FC_MM).sum(axis=0).mean())

print('=== v2.9 Post-training Diagnostics (wet/100% episode) ===')
print(f'Episode length:              {ep_len}  (expected 93)')
print(f'Mean irrigation mm/day:      {u_daily.mean():.2f}')
print(f'Spatial std (daily alloc):   {spatial_std:.4f}  (v2.7 s0: 0.002)')
print(f'corr(u_mean, rain_today):    {corr_rain:+.3f}  (v2.7 s0: -0.49)')
print(f'corr(u, x1) POOLED NxT:     {corr_x1_pooled:+.3f}  (v2.7 s0: +0.05 — want < 0)')
print(f'  (pooled N = {T*N:,} = {T} days x {N} agents)')
print(f'Waterlog days/agent:         {waterlog_days:.1f}  (v2.7 s0: 76.4; MPC: 20.2)')
print()
checks = [
    (ep_len == 93,          f'ep_len {ep_len} == 93'),
    (spatial_std > 0.01,    f'spatial_std {spatial_std:.4f} > 0.01'),
    (corr_rain < 0,         f'corr(u,rain) {corr_rain:+.3f} < 0'),
    (corr_x1_pooled < 0,    f'corr(u,x1) pooled {corr_x1_pooled:+.3f} < 0  (PRIMARY TARGET)'),
    (waterlog_days < 70,    f'waterlog_days {waterlog_days:.1f} < 70'),
]
for ok, label in checks:
    print(f'  {"✓" if ok else "⚠"} {label}')
print()
print('v2.7 seed 0 baseline for comparison:')
print('  corr(u,rain)=-0.489  corr(u,x1)=+0.048  waterlog=76.4d  spatial_std=0.002')

In [ ]:
# ── CELL 7: Resume from Drive checkpoint (if session was interrupted) ────────
# Fill in CHECKPOINT_STEP and CHECKPOINT_DRIVE_PATH, then uncomment and run.

# SEED = 0
# CHECKPOINT_STEP = 100_000
# CHECKPOINT_DRIVE_PATH = f'{DRIVE_ROOT}/sac_v29_seed{SEED}_YYYYMMDD_HHMMSS'
#
# import shutil, os
# local_dir = f'/content/thesis/results/rl/sac_v29_seed{SEED}'
# os.makedirs(local_dir, exist_ok=True)
# shutil.copytree(CHECKPOINT_DRIVE_PATH, local_dir, dirs_exist_ok=True)
#
# from stable_baselines3 import SAC
# from stable_baselines3.common.vec_env import DummyVecEnv
# from src.rl.gym_env import IrrigationEnv
# from src.rl.networks import V27CTDESACPolicy, make_sac_policy_kwargs
# from src.rl.train import (_make_lr_schedule, AdaptiveLRCallback,
#                            LR_START, LR_END, TOTAL_TIMESTEPS,
#                            GradClipCallback, RotatingReplayBufferCheckpoint,
#                            CRITIC_LOSS_SPIKE_THRESHOLD, CRITIC_LOSS_RECOVERY_THRESHOLD,
#                            LR_REDUCTION_FACTOR, LR_FLOOR, ROLLING_WINDOW, INTERVENTION_COOLDOWN_STEPS)
# from stable_baselines3.common.callbacks import (
#     CallbackList, CheckpointCallback, EvalCallback)
#
# env = DummyVecEnv([lambda: IrrigationEnv(randomize=True, curriculum_warmup_steps=0)])
# ckpt_zip = f'{local_dir}/checkpoints/sac_v29_seed{SEED}_{CHECKPOINT_STEP}_steps'
# model = SAC.load(ckpt_zip, env=env,
#                  custom_objects={'policy_class': V27CTDESACPolicy})
# lr_sched = _make_lr_schedule(LR_START, LR_END)
# model.lr_schedule = lr_sched
#
# eval_env = DummyVecEnv([lambda: IrrigationEnv(randomize=True, curriculum_warmup_steps=0)])
# eval_env.seed(SEED + 1000)
#
# callbacks = CallbackList([
#     EvalCallback(eval_env,
#                  best_model_save_path=f'{local_dir}/best_model',
#                  log_path=f'{local_dir}/eval_logs',
#                  eval_freq=25_000, n_eval_episodes=9, deterministic=True),
#     CheckpointCallback(save_freq=50_000,
#                        save_path=f'{local_dir}/checkpoints',
#                        name_prefix=f'sac_v29_seed{SEED}', verbose=1),
#     RotatingReplayBufferCheckpoint(save_freq=50_000, save_path=local_dir),
#     GradClipCallback(max_grad_norm=1.0),
#     AdaptiveLRCallback(lr_schedule=lr_sched,
#                        spike_threshold=CRITIC_LOSS_SPIKE_THRESHOLD,
#                        recovery_threshold=CRITIC_LOSS_RECOVERY_THRESHOLD,
#                        reduction_factor=LR_REDUCTION_FACTOR,
#                        lr_floor=LR_FLOOR, window=ROLLING_WINDOW,
#                        cooldown_steps=INTERVENTION_COOLDOWN_STEPS, verbose=0),
# ])
#
# remaining = TOTAL_TIMESTEPS - CHECKPOINT_STEP
# print(f'Resuming from step {CHECKPOINT_STEP}, {remaining} steps remaining...')
# model.learn(total_timesteps=remaining, reset_num_timesteps=False,
#             callback=callbacks, progress_bar=True)
# model.save(f'{local_dir}/sac_v29_seed{SEED}_final')
# print('Resume complete.')